# Sevastopol AI — бот (одна ячейка)

1. Впиши токен от **@BotFather** в ячейку ниже.
2. Нажми ▶️ — ячейка сама склонирует репозиторий, поставит зависимости и запустит бота.
3. Остановка — ■ (прервать ячейку). Colab засыпает без активности, поэтому для 24/7
   нужен сервер (раздел «Запуск 24/7» в README).

⚠️ Токен в ячейке виден всем, у кого есть доступ к ноутбуку. Не публикуй ноутбук
с вписанным токеном, а если он засветился — перевыпусти: @BotFather → /mybots →
API Token → Revoke.

In [ ]:
TG_TOKEN = "8927839211:AAGdKrfwB6bVlHcY9c0F5vUosy5IwKFnb2I"
# ⚠️ ВРЕМЕННО: токен вшит для теста в Colab и будет перевыпущен после него.
# ═══════════ Sevastopol AI — запуск бота одной ячейкой ═══════════

import os, pathlib, shutil, subprocess, sys

REPO = "https://github.com/Yurich-citycode/SevastopolAIbot.git"
DIR = "/content/SevastopolAIbot"


def sh(*cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


# 1) свежий код
if os.path.isdir(DIR):
    if subprocess.run(["git", "-C", DIR, "pull", "--ff-only", "origin", "main"]).returncode:
        shutil.rmtree(DIR)                      # локальные правки мешают — клонируем заново
if not os.path.isdir(DIR):
    sh("git", "clone", "--depth", "1", REPO, DIR)

# 2) зависимости
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", cwd=DIR)

# 3) настройки (файл .env в git не попадает)
env = pathlib.Path(DIR) / ".env"
env.write_text("\n".join([
    f"TG_TOKEN={TG_TOKEN}",
    "SPREADSHEET_ID=1RaHoS_8Ov-kNKSZJK015ceC6H3fsWnW-D-8Yee4ckQI",
    "ADMIN_ID=6106999216",
    "NEWS_CHANNEL_URL=https://t.me/Sevastopol_AI",
    "REFRESH_SECONDS=600",
    "STATE_FILE=bot_state.json",
]) + "\n", encoding="utf-8")
env.chmod(0o600)

# 4) запуск — процесс живёт, пока не нажмёшь ■
p = subprocess.Popen([sys.executable, "-u", "sevastopolaibot.py"], cwd=DIR,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("\n=== бот завершился с кодом", p.wait(), "===")